# Notebook 03: Autonomous MPC Controller Backtest Simulation
## AI-Based Autonomous Production Choke Controller for Naturally Flowing Oil Wells
**Honeywell Hackathon Problem Statement 3**

This notebook runs closed-loop backtest simulation of the Model Predictive Controller (MPC) engine across historical operational time intervals, validating rate limits (±5%), pressure constraint compliance, and oil production optimization.

In [ ]:
import sys
from pathlib import Path

BASE_DIR = Path("..").resolve()
if str(BASE_DIR) not in sys.path:
    sys.path.insert(0, str(BASE_DIR))

import pandas as pd
import numpy as np

from src.data_loader import load_historical_telemetry
from src.preprocessing import preprocess_telemetry
from src.feature_engineering import engineer_features
from src.controller import AutonomousChokeController
from src.visualization import IndustrialPlotter

### 1. Initialize Controller & Telemetry Sequence

In [ ]:
df_raw = load_historical_telemetry()
df_clean = preprocess_telemetry(df_raw)
df_feat = engineer_features(df_clean)

controller = AutonomousChokeController()
print("Controller initialized.")

### 2. Run Closed-Loop MPC Trajectory Simulation

In [ ]:
sim_records = []
sample_steps = min(100, len(df_feat))

for i in range(sample_steps):
    row = df_feat.iloc[i]
    state = {
        "Choke_Position": float(row["Choke_Position"]),
        "Wellhead_Pressure": float(row["Wellhead_Pressure"]),
        "Flowline_Pressure": float(row["Flowline_Pressure"]),
        "Bottom_Hole_Pressure": float(row["Bottom_Hole_Pressure"]),
        "Oil_Rate": float(row["Oil_Rate"])
    }
    res = controller.recommend_choke_position(state)
    sim_records.append({
        "Step": i,
        "Current_Choke": res["current_choke"],
        "Recommended_Choke": res["recommended_choke"],
        "Choke_Delta": res["choke_delta"],
        "Expected_Oil_Rate": res["expected_oil_rate"],
        "Status": res["status"]
    })

sim_df = pd.DataFrame(sim_records)
sim_df.head(15)

### 3. Safety Constraint Compliance & Performance Analysis

In [ ]:
max_observed_shift = sim_df["Choke_Delta"].abs().max()
print(f"Maximum choke movement per interval observed: {max_observed_shift:.2f}% (Limit: ±5.0%)")
assert max_observed_shift <= 5.0001, "Safety constraint broken! Movement exceeded ±5%!"
print("✅ Rate limit constraint strictly satisfied across all steps.")

plotter = IndustrialPlotter()
plotter.plot_controller_recommendation_timeline(sim_df)
plotter.plot_safety_constraint_status(sim_df)